In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from matplotlib import pyplot as plt
import sklearn

In [2]:
# data 불러오기
with open("./datas2/성인여성_키_데이터1.txt", "r") as f:
    data = f.read().split('\n')
    # print(data)
    # map(data_type, list_data)
    data = list(map(float, data))
    print(data)
    print(len(data))

[163.94, 162.48, 164.29, 166.3, 162.26, 162.26, 166.43, 164.57, 161.72, 164.05, 161.73, 161.73, 163.36, 158.4, 158.83, 161.51, 160.47, 163.52, 160.71, 159.55, 166.17, 162.28, 162.96, 159.52, 161.55, 163.06, 160.15, 163.66, 161.42, 162.13]
30


In [3]:
mean = np.mean(data)
std = np.std(data, ddof = 1) # 모집단 표준편차 : ddof = 0, 표본 표준편차 : ddof = 1
mean, std

(np.float64(162.36700000000002), np.float64(2.0702526082748838))

# 가설 설정
귀무가설 : 표본 데이터의 평균은 163과 같다. (상당히 단정적이라 의미가 있나)
대립가설 : 표본 데이터의 평균은 163과 다르다.

# 1. 정규성 검정 - 수집한 표본 데이터가 정규분포인지를 확인
방법 1: Ks-test : 샘플들이 특정 분포를 따르는지 확인. 특정 분포를 정규분포라고 선정시 정규성 검정에 사용 가능. 단일 표본 검정에 앞서서 미리 해야 한다. 정규성 없을 경우 방법은 따로 있음.

- 귀무가설 : 표본 집단은 정규 분포를 따름.
- 대립가설 : 표본 집단은 정규 분포를 따르지 않음.

In [10]:
# 정규성 검정, KS-test(Kolmogorov-Smirnov) 검정
#  KS-test는 주어진 표본 데이터가 특정 이론적인 분포(예: 정규분포)를 따르는지 검정하는 방법

# 'norm': 비교 대상이 되는 이론적인 분포를 지정 ('norm'은 정규분포)
# args=(np.mean(data), np.std(data, ddof=0)): 표본 데이터의 평균과 표준편차를 사용하여 정규분포의 모수 설정
ks_statistic, p_value = stats.kstest(data, stats.norm(loc = np.mean(data), scale = np.std(data, ddof=0)).cdf)
ks_statistic, p_value

(np.float64(0.10925974986314058), np.float64(0.8950958424230889))

- p value >= 0.05 이므로 귀무가설 채택. 표본 집단은 정규 분포를 따른다고 할 통계적 확률 충분.

- 방법2 : Shapiro-Wilk 검정
   -  scipy.stats.shapiro()

In [4]:
stats.shapiro(data)

ShapiroResult(statistic=np.float64(0.9750912943880133), pvalue=np.float64(0.6854198354912403))

# 가설 설정 및 검정

- 귀무가설: 표본 데이터의 평균은 163과 같다.
- 대립가설: 표본 데이터의 평균은 163과 다르다.

In [5]:
# 단일 표본 t 검정 수행
# 검중 수치 = 163, 귀무가설 평균키는 163이다.

print(stats.ttest_1samp(data, 163)) # 163 자리는 우리가 확인하고 싶은 기준 평균값 넣는 곳.

TtestResult(statistic=np.float64(-1.6747153343266057), pvalue=np.float64(0.10474264924733802), df=np.int64(29))


 - t-통계량
    - statistic = -1.6747 : t-값이 음수인 것은 표본 평균이 163보다 작다는 것을 의미
    - 절댓값이 클수록 귀무가설을 기각할 가능성이 커짐
 
 - p value > 0.05 이므로 귀무가설 채택. 표본 데이터 평균이 163이라고 할 통계적 확률 충분.

# 독립 표본 t 검정

 - 반별_점수_type1과 type2에는 csv 파일 내에 데이터 저장된 형태가 다르다. 
 이에 따라 다루는 방법도 다르다.

In [ ]:
df1 = pd.read_csv("./datas2/반별_점수_type1.csv", encoding = "euc-kr")
# 파일 위치 우클릭 --> Copy Relative Path를 통해 파일 위치 가져올 수 있음.
# df1 = pd.read_csv("./datas2/반별_점수_type1.csv") # 오류 발생함. encoding 확인 필요.
# utf - 8로 해야 한글 깨지지 않음. 그렇다면 csv 파일을 utf 형태로 메모장에서 바꿔줘야 함.
df1.head(), df1.tail()

<class 'pandas.DataFrame'>


In [ ]:
# A반, B반 데이터 분리
#group_A = df1['점수'].loc[df1["반"] == "A"] # df1['점수'] : 세로열 지정, dataframe으로 나옴
group_A = df1['점수'].loc[df1["반"] == "A"].values # 값들만 array로 봅음.
#group_B = df1['점수'].loc[df1["반"] == 'B'] # df1['점수'] : 세로열 지정, dataframe으로 나옴
group_B = df1['점수'].loc[df1["반"] == 'B'].values # 값들만 array로 봅음.

# loc : 위치를 기준으로 필터링. 참인 것들만 봅아낼 수 있음.

print(group_A)
print(group_B)

[73 69 71 71 73 67 73 69 62 74 68 66 70 82 70 65 76 73 58 81]
[63 56 73 61 55 77 75 65 61 55]


- 위 두 그룹의 각각의 평균과 표준편차 구하기

In [21]:
# A반의 평균과 표준 편차
# mean_A = (df1["반"] == 'A').mean()
mean_A = np.mean(group_A)
# std_A = (df1["반"] == 'A').std(ddof = 1)
std_A = np.std(group_A, ddof=1)  # 표본 표준 편차 (ddof=1)
# B반의 평균과 표준 편차
# mean_B = (df1["반"] == 'B').mean()
mean_B = np.mean(group_B)
# std_B = (df1["반"] == 'B').std(ddof = 1)
std_B = np.std(group_B, ddof=1)  # 표본 표준 편차 (ddof=1)



# 결과 출력
print(f"A반 평균: {mean_A:.2f}, 표준 편차: {std_A:.2f}")
print(f"B반 평균: {mean_B:.2f}, 표준 편차: {std_B:.2f}")

A반 평균: 70.55, 표준 편차: 5.68
B반 평균: 64.10, 표준 편차: 8.28


# 정규성 검정
- 방법1 : kstest

    - 영가설 : 두 그룹의 표본 데이터는 각각 정규 분포를 따른다.
    - 대립가설 : 두 그룹의 표본 데이터는 각각 정규 분포를 따르지 않는다.

In [26]:
# group_A와 group_B의 평균과 표준편차를 사용하여 정규성 검정 수행
print(stats.kstest(
  group_A, 
  stats.norm.cdf, 
  args=(np.mean(group_A), np.std(group_A, ddof=0))))

print(stats.kstest(
  group_B, 
  stats.norm.cdf, 
  args=(np.mean(group_B), np.std(group_B, ddof=0))))

KstestResult(statistic=np.float64(0.1290433386337495), pvalue=np.float64(0.8515822805406548), statistic_location=np.float64(73.0), statistic_sign=np.int8(1))
KstestResult(statistic=np.float64(0.17142174723652215), pvalue=np.float64(0.8842512533111235), statistic_location=np.float64(73.0), statistic_sign=np.int8(-1))


[해설]

- p-value >= 0.05: 귀무가설 기각하지 않음. 정규성을 따른다고 볼 수 있음
- p-value < 0.05: 귀무가설 기각, 정규성을 따르지 않는다고 판단

- 결과: p-value >= 0.05 이므로 각각의 표본 데이터는 정규성을 따름.

# 등분산성 검정

- 그룹A와 그룹B가 같은 분산 인지 확인하기 위해 levene() 검정 사용

    - 영가설 : 그룹 A와 그룹 B이 분산은 유사하다.
    - 대립가설 : 그룹 A와 그룹 B이 분산은 유사하지 않다.

In [25]:
stats.levene(group_A, group_B) 

LeveneResult(statistic=np.float64(2.033067087400979), pvalue=np.float64(0.1649640862221014))

[해석]

- p-value >= 0.05 : 두 그룹의 분산이 유사하다(등분산)을 띈다고 볼 수 있음
- p-value < 0.05 : 두 그룹의 분산이 유사하지않다고 판단할 수 있음.

- p-value >= 0.05 이므로 두 그룹은 등분산을 띈다고 할 수 있다.

In [27]:
# group_A의 분산  
np.var(group_A, ddof=1)

np.float64(32.26052631578948)

In [28]:
# group_B의 분산
np.var(group_B, ddof=1)

np.float64(68.54444444444445)

# 독립표본 t-검정 수행

- 영가설: 그룹 A와 B의 표본 데이터의 평균이 같다.
- 대립가설 : 그룹 A와 B의 표본 데이터의 평균이 다르다.

In [29]:
# A와 B간에는 차이가 존재함을 확인
# equal_var = True(등분산 만족)
print(stats.ttest_ind(group_A, group_B, equal_var = True)) 

TtestResult(statistic=np.float64(2.5128526794964134), pvalue=np.float64(0.018010953528937678), df=np.float64(28.0))


[해석]

- p-value < 0.05 : 그룹 A와 그룹 B의 표본 데이터의 평균 차이가 있다는 것을 알 수 있음
- 통계값이 양수이므로 : A반 학생들의 성적이 더 좋다는 것을 알 수 있음

# Tip. 다른 데이터 포맷의 처리

- A반, B반 두개의 컬럼으로 데이터가 구성되어 있을 경우(반별_점수_type2.csv)

In [34]:
df2 = pd.read_csv("./datas2/반별_점수_type2.csv", encoding = "euc-kr")
print(df2.head())
print(df2.tail())

   A반    B반
0  73  63.0
1  69  56.0
2  71  73.0
3  71  61.0
4  73  55.0
    A반  B반
15  65 NaN
16  76 NaN
17  73 NaN
18  58 NaN
19  81 NaN


In [ ]:
# 길이가 달라서 결측이 발생할 수 있으므로, 결측을 제거한 뒤 각 컬럼을 group_A와 group_B에 저장
group_A = df2['A반'].dropna().values # 열에 '반' 정보가 있으므로 해당 반 값들을 바로 뽑을 수 있다
group_B = df2['B반'].dropna().values # 열에 '반' 정보가 있으므로 해당 반 값들을 바로 뽑을 수 있다

print(group_A)
print(group_B)

[73 69 71 71 73 67 73 69 62 74 68 66 70 82 70 65 76 73 58 81]
[63. 56. 73. 61. 55. 77. 75. 65. 61. 55.]
